In [1]:
import pandas as pd
import numpy as np

In [2]:
#define a population for S and I people with high and low activities across three counties
pop_size = pd.DataFrame({
    'InfState': ['S', 'S', 'I', 'I', 'S', 'S', 'I', 'I', 'S', 'S', 'I', 'I'],
    'County': pd.Categorical(['Orange_HA', 'Orange_LA', 'Orange_HA', 'Orange_LA', 
                              'Durham_HA', 'Durham_LA', 'Durham_HA', 'Durham_LA', 
                              'Chatham_HA', 'Chatham_LA', 'Chatham_HA', 'Chatham_LA'], 
                             categories=['Orange_HA', 'Orange_LA', 'Durham_HA', 'Durham_LA', 'Chatham_HA', 'Chatham_LA']),
    'N': [5000, 5000, 8000, 316, 40000, 50000, 2000, 280, 10000, 20000, 8000, 266],
    'T': 2025
})
pop_size

,InfState,County,N,T
0,S,Orange_HA,5000,2025
1,S,Orange_LA,5000,2025
2,I,Orange_HA,8000,2025
3,I,Orange_LA,316,2025
4,S,Durham_HA,40000,2025
5,S,Durham_LA,50000,2025
6,I,Durham_HA,2000,2025
7,I,Durham_LA,280,2025
8,S,Chatham_HA,10000,2025
9,S,Chatham_LA,20000,2025


In [3]:
#contact rate matrix, Beta: High Activity, Low Activity
contact_rate_matrix = np.array([[0.1428, 0.0049], [0.00055, 0.0077]])
contact_rate_matrix

array([[0.1428 , 0.0049 ],
       [0.00055, 0.0077 ]])

In [4]:
#Geographcial mixing matrix, T: Orange, Durham, Chatham
geo_mixing_matrix = np.array([[0.7, 0.1869, 0.1131], [0.1851, 0.7, 0.1150], [0.2210, 0.07895, 0.7]])
geo_mixing_matrix

array([[0.7    , 0.1869 , 0.1131 ],
       [0.1851 , 0.7    , 0.115  ],
       [0.221  , 0.07895, 0.7    ]])

In [5]:
#transmission rate of high and low activities by geo
transmission_by_geo = np.kron(geo_mixing_matrix, contact_rate_matrix)
transmission_by_geo = np.round(transmission_by_geo, decimals=5)
transmission_by_geo

array([[9.996e-02, 3.430e-03, 2.669e-02, 9.200e-04, 1.615e-02, 5.500e-04],
       [3.800e-04, 5.390e-03, 1.000e-04, 1.440e-03, 6.000e-05, 8.700e-04],
       [2.643e-02, 9.100e-04, 9.996e-02, 3.430e-03, 1.642e-02, 5.600e-04],
       [1.000e-04, 1.430e-03, 3.800e-04, 5.390e-03, 6.000e-05, 8.900e-04],
       [3.156e-02, 1.080e-03, 1.127e-02, 3.900e-04, 9.996e-02, 3.430e-03],
       [1.200e-04, 1.700e-03, 4.000e-05, 6.100e-04, 3.800e-04, 5.390e-03]])

In [6]:
index_names= ['Orange_HA', 'Orange_LA', 'Durham_HA', 'Durham_LA', 'Chatham_HA', 'Chatham_LA']
column_names = ['Orange_HA', 'Orange_LA', 'Durham_HA', 'Durham_LA', 'Chatham_HA', 'Chatham_LA']
pd.DataFrame(transmission_by_geo, index=index_names, columns=column_names)

,Orange_HA,Orange_LA,Durham_HA,Durham_LA,Chatham_HA,Chatham_LA
Orange_HA,0.09996,0.00343,0.02669,0.00092,0.01615,0.00055
Orange_LA,0.00038,0.00539,0.00010,0.00144,0.00006,0.00087
Durham_HA,0.02643,0.00091,0.09996,0.00343,0.01642,0.00056
Durham_LA,0.00010,0.00143,0.00038,0.00539,0.00006,0.00089
Chatham_HA,0.03156,0.00108,0.01127,0.00039,0.09996,0.00343
Chatham_LA,0.00012,0.00170,0.00004,0.00061,0.00038,0.00539


In [7]:
#infection array
inf_array = pop_size.loc[pop_size['InfState']=='I'].groupby(['County'], observed=False)['N'].sum(numeric_only=True).values
inf_array

array([8000,  316, 2000,  280, 8000,  266])

In [8]:
#probability of infectino
prI = np.power(np.exp(-1 * transmission_by_geo), inf_array)
prI

array([[0.00000000e+000, 3.38280448e-001, 6.56690231e-024,
        7.72904332e-001, 7.74734575e-057, 8.63898495e-001],
       [4.78348895e-002, 1.82092587e-001, 8.18730753e-001,
        6.68178450e-001, 6.18783392e-001, 7.93406165e-001],
       [1.48858880e-092, 7.50091560e-001, 1.49915721e-087,
        3.82739759e-001, 8.93463586e-058, 8.61603578e-001],
       [4.49328964e-001, 6.36430537e-001, 4.67666427e-001,
        2.21086777e-001, 6.18783392e-001, 7.89196452e-001],
       [2.23526598e-110, 7.10859840e-001, 1.62555766e-010,
        8.96551089e-001, 0.00000000e+000, 4.01567356e-001],
       [3.82892886e-001, 5.84382234e-001, 9.23116346e-001,
        8.42990155e-001, 4.78348895e-002, 2.38415578e-001]])

In [9]:
prI = 1 - prI.prod(axis=1) #using 1- prI is questionable, maybe we should just use prI, see prI_2 below for different math
prI

array([1.        , 0.9976606 , 1.        , 0.98556097, 1.        ,
       0.99801421])

In [10]:
#suspectible mask based on initial population
is_susceptible = (pop_size['InfState'] == 'S')
is_susceptible

0      True
1      True
2     False
3     False
4      True
5      True
6     False
7     False
8      True
9      True
10    False
11    False
Name: InfState, dtype: bool

In [11]:
#number of susceptible people with high and low activities in each county
deltas = pop_size[is_susceptible].copy()
deltas

,InfState,County,N,T
0,S,Orange_HA,5000,2025
1,S,Orange_LA,5000,2025
4,S,Durham_HA,40000,2025
5,S,Durham_LA,50000,2025
8,S,Chatham_HA,10000,2025
9,S,Chatham_LA,20000,2025


In [12]:
present_category_codes = pop_size['County'].cat.codes.to_numpy()
present_category_codes

array([0, 1, 0, 1, 2, 3, 2, 3, 4, 5, 4, 5], dtype=int8)

In [13]:
susceptible_group_codes = present_category_codes[is_susceptible.to_numpy()]
susceptible_group_codes

array([0, 1, 2, 3, 4, 5], dtype=int8)

In [14]:
#prI of each group, i.e. prI for high and low activities of each county
prI_per_group = prI[susceptible_group_codes]
prI_per_group

array([1.        , 0.9976606 , 1.        , 0.98556097, 1.        ,
       0.99801421])

In [15]:
#infected people with high and low activities in each county
deltas['N'] = -deltas['N'] * prI_per_group
deltas

,InfState,County,N,T
0,S,Orange_HA,-5000.000000,2025
1,S,Orange_LA,-4988.302991,2025
4,S,Durham_HA,-40000.000000,2025
5,S,Durham_LA,-49278.048267,2025
8,S,Chatham_HA,-10000.000000,2025
9,S,Chatham_LA,-19960.284261,2025


In [16]:
prI_2 = np.power(np.exp(-1 * transmission_by_geo), inf_array)
prI_2 = prI_2.prod(axis=1)
prI_2

array([0.00000000e+000, 2.33940185e-003, 4.93201210e-237, 1.44390347e-002,
       0.00000000e+000, 1.98578694e-003])

In [17]:
-deltas['N'] * prI_2

0     0.000000e+00
1     1.166965e+01
4    1.972805e-232
5     7.115274e+02
8     0.000000e+00
9     3.963687e+01
Name: N, dtype: float64